# Hệ thống ghép ảnh UAV nhận biết rủi ro: Dự báo lỗi, Định tuyến và Phục hồi (Failure-Aware Mosaicking Pipeline)

### 1. Đặt vấn đề và mục tiêu nghiên cứu
Khi ghép video hành trình từ máy bay không người lái (UAV) thành ảnh toàn cảnh (mosaic):
- Ở các khu vực địa hình đồng nhất hoặc lặp lại (đồng cỏ, mặt nước, mái tôn), các bộ so khớp đặc trưng dễ sinh ra kết quả sai lệch hình học (geometric registration failure).
- Do không có thiết bị đo đạc mặt đất chuyên dụng (RTK/GCP) làm ground truth tuyệt đối ngoài thực địa, hệ thống cần cơ chế tự giám sát dựa trên tính nhất quán và các chỉ báo quang học/hình học tính toán nhanh.
- Khi phát hiện rủi ro, hệ thống cần cơ chế định tuyến (matcher routing) và phục hồi sự cố (recovery policies) để tránh làm hỏng toàn bộ chuỗi ghép.

Notebook này triển khai các thực nghiệm kỹ thuật tương ứng với 3 câu hỏi nghiên cứu (Research Questions - RQ) theo quyết định `RDR-0003`, `RDR-0007`, `RDR-0008` (Proposed) và `RDR-0011` (Accepted):

| Câu hỏi nghiên cứu | Mục tiêu kỹ thuật | Phương pháp thực nghiệm trong notebook |
| :--- | :--- | :--- |
| **RQ1: Dự báo lỗi (Failure Prediction)** | Đánh giá khả năng cảnh báo sớm của các chỉ báo tính nhanh (Spearman rho & ROC-AUC). | So sánh 8 chỉ báo tính toán nhanh với sai số bước ghép đo trên matcher tham chiếu chất lượng cao (SIFT 4000 đặc trưng, ảnh gốc, cổng inlier $\ge 100$). |
| **RQ2: Phân luồng thích ứng (Matcher Routing)** | Đánh giá tiềm năng tiết kiệm thời gian của cơ chế phân luồng 2 tầng. | Mô phỏng chính sách phân luồng trên nhật ký đo được: mặc định dùng matcher nhẹ (SIFT), chỉ leo thang sang matcher mạnh (LightGlue) hoặc ảnh gốc khi chỉ báo vượt ngưỡng rủi ro $\tau$. |
| **RQ3: Chiến lược phục hồi (Recovery Policies)** | Đánh giá hành vi của 4 chiến lược khi matcher gặp tình huống rủi ro. | Đánh giá 4 chiến lược: Mặc kệ (`none`), Bỏ qua (`drop`), Ghép lùi (`anchor`), Tách mảng (`split`) trên cùng ngưỡng kích hoạt, đo độ trôi nội bộ từng mảnh và tỷ lệ khung hình giữ lại. |

### 2. Phạm vi thực nghiệm và giao thức đánh giá
- **Giao thức LOSO (Leave-One-Sequence-Out):** Theo `RDR-0011`, quy trình đánh giá chuẩn chia theo sequence, hiệu chỉnh ngưỡng trên tập train và kiểm thử trên sequence giữ lại. Các ngưỡng số trị trong notebook này (như $\tau_{ncc} = 0,93$, tolerance 2 px) mang tính chất **khám phá (exploratory)** theo `RDR-0008` (Proposed), chưa phải thông số chốt sau cùng.
- **Cửa sổ quét thử nghiệm:** Mặc định chạy trên 2 cửa sổ chọn lọc (`phantom3-village-kfs` frame 80–125 và `phantom3-grass-kfs` frame 0–45) đại diện cho các chế độ lỗi khác nhau. Cấu hình hỗ trợ quét toàn bộ frame của mọi sequence có trong thư mục dữ liệu (`EXECUTION_MODE="full"`).
- **Cổng kiểm soát chất lượng tham chiếu:** Theo `RDR-0007`, bước có cặp tham chiếu dưới 100 inliers hoặc RANSAC thất bại được ghi nhận là không khả dụng (`valid = False`), loại khỏi RQ1 và không tạo chuỗi tham chiếu giả định.


In [ ]:
# ==============================================================================
# CẤU HÌNH NHANH (QUICK CONFIG)
# Đổi "pilot" thành "full" để chạy toàn bộ dataset; giữ "pilot" để thử nghiệm nhanh.
# ==============================================================================
EXECUTION_MODE = "pilot"  # "pilot" hoặc "full"

import os
os.environ["UAV_EXECUTION_MODE"] = EXECUTION_MODE

import hashlib
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib
import numpy as np
import torch
from IPython.display import Markdown, display

# 1. Kiểm tra môi trường thực thi (Cloud Guard) per RULES.md §8 & RDR-0010
is_colab = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and Path("/kaggle").is_dir()
if not (is_colab or is_kaggle):
    raise RuntimeError(
        "LOCAL EXECUTION PROHIBITED per RULES.md §8 and RDR-0010. "
        "Experiments must run exclusively on hosted Google Colab or Kaggle cloud runtimes. "
        "Personal computers are restricted to static code review and JSON verification."
    )

# Google Drive mount (Colab) - fail closed nếu không mount được
if is_colab:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.ismount("/content/drive") or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive mount failed; persistent storage unavailable.")

# 2. Cài đặt dependencies nếu thiếu
to_install = []
try:
    import lightglue
except ImportError:
    to_install.append("git+https://github.com/cvg/LightGlue.git@eb42fee2d71449efb0aa5c10549752b5d75384d8")
try:
    import transformers
except ImportError:
    to_install.append("transformers==4.57.6")
if to_install:
    print(f"Installing missing dependencies: {to_install}...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)

try:
    get_ipython  # noqa: B018
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402


def resolve_paths():
    """Một root dữ liệu rõ ràng; output Colab bắt buộc nằm trên Drive đã mount."""
    default_project = "/content/drive/MyDrive/mosaicking_from_uav_video_research" if is_colab else "/kaggle/working"
    project_dir = Path(os.environ.get("UAV_PROJECT_ROOT", default_project)).expanduser().resolve()
    if not project_dir.is_dir():
        raise FileNotFoundError(f"Project directory missing: {project_dir}")
    data_dir = next((project_dir / name for name in ("dataset", "data")
                     if (project_dir / name).is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError(f"Expected dataset/ or data/ inside {project_dir}")
    if is_kaggle and not os.environ.get("UAV_OUTPUT_ROOT"):
        raise ValueError("Set UAV_OUTPUT_ROOT and save Kaggle version/output for persistent handoff.")
    default_output = "/content/drive/MyDrive/mosaicking_from_uav_video_research/outputs"
    output_root = Path(os.environ.get("UAV_OUTPUT_ROOT", default_output)).expanduser().resolve()
    if is_colab and not output_root.is_relative_to(Path("/content/drive").resolve()):
        raise ValueError(f"Colab output must be on mounted Google Drive: {output_root}")
    return project_dir, data_dir, output_root


ROOT, DATA_DIR, OUTPUT_ROOT = resolve_paths()


def ensure_sequence(data_dir: Path, seq_name: str):
    """Đảm bảo sequence có thư mục ảnh; tự động giải nén an toàn nếu có archive."""
    target = data_dir / seq_name
    rgb_dirs = list(target.rglob("rgb"))
    if rgb_dirs and any(rgb_dirs[0].glob("*.jpg")):
        return target
    import tarfile, zipfile
    for ext in (".tar.gz", ".tgz", ".zip"):
        archive = data_dir / f"{seq_name}{ext}"
        if archive.is_file():
            print(f"Giải nén an toàn {archive.name}...")
            target.mkdir(parents=True, exist_ok=True)
            dest_dir = target.resolve()
            if ext == ".zip":
                with zipfile.ZipFile(archive, "r") as z:
                    for member in z.infolist():
                        dest = (target / member.filename).resolve()
                        if not dest.is_relative_to(dest_dir):
                            raise RuntimeError(f"Unsafe zip member: {member.filename}")
                    z.extractall(target)
            else:
                with tarfile.open(archive, "r:*") as t:
                    if not hasattr(tarfile, "data_filter"):
                        raise RuntimeError("Safe tar extraction requires Python with tarfile.data_filter.")
                    t.extractall(target, filter="data")
            break
    return target


# Cấu hình chế độ chạy và khám phá sequence
EXECUTION_MODE = os.environ.get("UAV_EXECUTION_MODE", EXECUTION_MODE).lower()
if EXECUTION_MODE not in ("pilot", "full"):
    raise ValueError("UAV_EXECUTION_MODE must be pilot or full")
DEFAULT_WINDOWS = {"phantom3-village-kfs": (80, 45), "phantom3-grass-kfs": (0, 45)}
discovered = set()
for item in DATA_DIR.iterdir():
    if item.is_dir() and any(item.rglob("rgb")):
        discovered.add(item.name)
    for suffix in (".tar.gz", ".tgz", ".zip"):
        if item.is_file() and item.name.endswith(suffix):
            discovered.add(item.name[:-len(suffix)])
if EXECUTION_MODE == "full":
    if not discovered:
        raise FileNotFoundError(f"No sequences found in {DATA_DIR}")
    SEQUENCES = tuple(sorted(discovered))
    WINDOWS = {s: (0, None) for s in SEQUENCES}
else:
    SEQUENCES = tuple(DEFAULT_WINDOWS)
    WINDOWS = dict(DEFAULT_WINDOWS)
print(f"Execution mode: {EXECUTION_MODE}; sequences: {SEQUENCES}")

for s in SEQUENCES:
    ensure_sequence(DATA_DIR, s)

MAX_SIDE = 640
MAX_KEYPOINTS = 2000
RANSAC_PX = 3.0
RATIO = 0.75
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MIN_INLIERS = 100          # Cổng hợp lệ matcher tham chiếu theo RDR-0007
TOLERANCE_PX = 2.0         # Mức nền nhiễu SIFT tham chiếu theo RDR-0008 (Proposed)
TAU_NCC = 0.95             # Ngưỡng NCC mặc định khám phá
TAU_INLIER_RATIO = 0.30    # Ngưỡng tỷ lệ inlier mặc định khám phá
TAU_GRID = (0.90, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99)
ESCALATION_BUDGET = 0.25
ANCHOR_BACK = 3
CHEAP = "sift_bf"
STRONG = "superpoint_lightglue"
DIAGNOSTIC = "efficientloftr"
FULLRES = "superpoint_lightglue@full"
POLICIES = ("none", "drop", "anchor", "split")

# Cấu hình matcher cho RQ3 theo từng sequence (thay vì cố định 2-entry zip)
RQ3_MATCHER_MAP = {
    "phantom3-village-kfs": CHEAP,
    "phantom3-grass-kfs": DIAGNOSTIC,
}

print("Root:", ROOT, "| Data:", DATA_DIR, "| Output:", OUTPUT_ROOT)
print("Torch:", torch.__version__, "| OpenCV:", cv2.__version__, "| Device:", DEVICE)

In [ ]:
# Tải ảnh ở độ phân giải làm việc và độ phân giải gốc với bộ nhớ đệm có giới hạn (bounded cache).
class LazyProcessedImages:
    """Bộ nạp ảnh resize theo nhu cầu, lưu tối đa max_cached ảnh trong RAM để tránh OOM khi chạy full mode."""

    def __init__(self, files, max_side=MAX_SIDE, max_cached=32):
        self.files = list(files)
        self.max_side = max_side
        self.max_cached = max_cached
        self.cache = {}
        self.order = []
        self.processed_size = None
        self.original_size = None
        if self.files:
            # Đọc kích thước frame đầu tiên để xác định shape
            bgr = cv2.imread(str(self.files[0]))
            if bgr is None:
                raise ValueError(f"không đọc được {self.files[0]}")
            self.original_size = (bgr.shape[1], bgr.shape[0])
            scale = min(1.0, self.max_side / max(bgr.shape[:2]))
            self.processed_size = (max(32, int(bgr.shape[1] * scale) // 32 * 32),
                                  max(32, int(bgr.shape[0] * scale) // 32 * 32))

    def __getitem__(self, index):
        if index not in self.cache:
            bgr = cv2.imread(str(self.files[index]))
            if bgr is None:
                raise ValueError(f"không đọc được {self.files[index]}")
            if len(self.cache) >= self.max_cached:
                oldest = self.order.pop(0)
                self.cache.pop(oldest, None)
            rgb = cv2.cvtColor(cv2.resize(bgr, self.processed_size, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
            self.cache[index] = rgb
            self.order.append(index)
        return self.cache[index]

    def __len__(self):
        return len(self.files)


def frames_of(sequence, start, count, max_side=MAX_SIDE):
    folders = list((DATA_DIR / sequence).rglob("rgb"))
    if len(folders) != 1:
        raise ValueError(f"cần đúng một thư mục rgb: {sequence}")
    all_files = sorted(folders[0].glob("*.jpg"), key=lambda p: float(p.stem))[start:]
    if count is not None and count > 0:
        files = all_files[:count + 1]
    else:
        files = all_files
    if len(files) < 2:
        raise ValueError(f"{sequence}: chỉ có {len(files)} frame cho start={start}, count={count}")
    lazy_images = LazyProcessedImages(files, max_side=max_side)
    return files, lazy_images, lazy_images.original_size


def to_original(H, processed, original):
    """H ở hệ pixel ảnh resize -> hệ pixel ảnh gốc."""
    D = np.diag([original[0] / processed[0], original[1] / processed[1], 1.0])
    return D @ H @ np.linalg.inv(D)


class LazyImages:
    """Danh sách ảnh gốc với bộ nhớ đệm có giới hạn (bounded cache) tránh tràn RAM."""

    def __init__(self, files, max_cached=16):
        self.files = list(files)
        self.max_cached = max_cached
        self.cache = {}
        self.order = []

    def __getitem__(self, index):
        if index not in self.cache:
            bgr = cv2.imread(str(self.files[index]))
            if bgr is None:
                raise ValueError(f"không đọc được {self.files[index]}")
            if len(self.cache) >= self.max_cached:
                oldest = self.order.pop(0)
                self.cache.pop(oldest, None)
            self.cache[index] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            self.order.append(index)
        return self.cache[index]

    def __len__(self):
        return len(self.files)


def dump(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def fmt(value, width, digits=2):
    """In số an toàn: giá trị có thể là None (chưa đo được) thì ghi 'n/a'."""
    return "n/a".rjust(width) if value is None else f"{value:{width}.{digits}f}"


In [ ]:
# Adapter matcher - giữ nguyên giao diện của benchmark chuẩn hoá để dùng chung.
def tensor(image, device):
    return torch.from_numpy(image.copy()).permute(2, 0, 1).float().to(device) / 255


def load_matcher(method, device, max_points=MAX_KEYPOINTS):
    if method == CHEAP:
        detector = cv2.SIFT_create(nfeatures=max_points)
        matcher = cv2.BFMatcher(cv2.NORM_L2)

        def match(a, b):
            detected = [detector.detectAndCompute(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), None) for img in (a, b)]
            (k0, d0), (k1, d1) = detected
            if d0 is None or d1 is None or len(d1) < 2:
                return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)
            good = [m for pair in matcher.knnMatch(d0, d1, k=2) if len(pair) == 2
                    for m, n in [pair] if m.distance < RATIO * n.distance]
            return (np.float32([k0[m.queryIdx].pt for m in good]).reshape(-1, 2),
                    np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 2))
        return match
    if method == STRONG:
        from lightglue import LightGlue, SuperPoint
        from lightglue.utils import rbd
        extractor = SuperPoint(max_num_keypoints=max_points).eval().to(device)
        matcher = LightGlue(features="superpoint").eval().to(device)
        extractor.requires_grad_(False)
        matcher.requires_grad_(False)

        @torch.inference_mode()
        def match(a, b):
            features = [extractor.extract(tensor(img, device), resize=None) for img in (a, b)]
            if min(f["keypoints"].shape[1] for f in features) == 0:
                return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)
            index = rbd(matcher({"image0": features[0], "image1": features[1]}))["matches"]
            return (rbd(features[0])["keypoints"][index[:, 0]].cpu().numpy(),
                    rbd(features[1])["keypoints"][index[:, 1]].cpu().numpy())
        return match
    if method == DIAGNOSTIC:
        from transformers import AutoImageProcessor, AutoModelForKeypointMatching
        source = "zju-community/efficientloftr"
        revision = "face1a79050ffa3e9da28720d1cf93aaf2e8f421"
        processor = AutoImageProcessor.from_pretrained(source, revision=revision)
        model = AutoModelForKeypointMatching.from_pretrained(source, revision=revision).eval().to(device)
        model.requires_grad_(False)

        @torch.inference_mode()
        def match(a, b):
            inputs = processor([a, b], do_resize=False, return_tensors="pt").to(device)
            out = processor.post_process_keypoint_matching(
                model(**inputs), [[a.shape[:2], b.shape[:2]]], threshold=0.2)[0]
            return out["keypoints0"].cpu().numpy(), out["keypoints1"].cpu().numpy()
        return match
    raise ValueError(method)


In [ ]:
# Matcher tham chiếu (RDR-0007) + hình học + chỉ báo rẻ.
REFERENCE_SIFT = cv2.SIFT_create(nfeatures=4000)
REFERENCE_BF = cv2.BFMatcher(cv2.NORM_L2)
_REFERENCE_CACHE = {}
_MATCH_CACHE = {}


def sync_device():
    """Đồng bộ CUDA trước khi bấm giờ để đo lường công bằng không lẫn hàng đợi GPU."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reference_pair(path_a, path_b):
    """H của cặp (b -> a) ở ẢNH GỐC bằng SIFT 4000 đặc trưng; có cache.

    Trả về (H_ref, inliers, matches, seconds, valid).
    Khi thất bại (ảnh rỗng, không đủ đặc trưng, RANSAC hỏng), trả về H_ref = None và valid = False,
    không ném ngoại lệ làm ngắt toàn bộ run thực nghiệm.
    """
    key = (str(path_a), str(path_b))
    if key in _REFERENCE_CACHE:
        return _REFERENCE_CACHE[key]
    started = time.perf_counter()
    grays = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in (path_a, path_b)]
    if any(g is None for g in grays):
        res = (None, 0, 0, time.perf_counter() - started, False)
        _REFERENCE_CACHE[key] = res
        return res
    detected = [REFERENCE_SIFT.detectAndCompute(g, None) for g in grays]
    (k0, d0), (k1, d1) = detected
    if d0 is None or d1 is None or len(d0) < 2 or len(d1) < 2:
        res = (None, 0, 0, time.perf_counter() - started, False)
        _REFERENCE_CACHE[key] = res
        return res
    good = [m for pair in REFERENCE_BF.knnMatch(d0, d1, k=2) if len(pair) == 2
            for m, n in [pair] if m.distance < RATIO * n.distance]
    if len(good) < 4:
        res = (None, 0, len(good), time.perf_counter() - started, False)
        _REFERENCE_CACHE[key] = res
        return res
    p0 = np.float32([k0[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    p1 = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    cv2.setRNGSeed(0)
    H, mask = cv2.findHomography(p1, p0, cv2.RANSAC, RANSAC_PX)
    elapsed = time.perf_counter() - started
    if H is None or mask is None or not np.isfinite(H).all() or np.linalg.matrix_rank(H) != 3 or abs(H[2, 2]) < 1e-9:
        res = (None, 0, len(good), elapsed, False)
        _REFERENCE_CACHE[key] = res
        return res
    H_norm = H / H[2, 2]
    inliers = int(mask.sum())
    valid = inliers >= MIN_INLIERS
    res = (H_norm, inliers, len(good), elapsed, valid)
    _REFERENCE_CACHE[key] = res
    return res


def geometry(p0, p1):
    """H ánh xạ p1 -> p0 (frame hiện tại -> frame trước), RANSAC cố định seed."""
    if len(p0) < 4 or len(p1) < 4:
        return None, np.zeros(len(p0), bool), None
    cv2.setRNGSeed(0)
    H, mask = cv2.findHomography(p1, p0, cv2.RANSAC, RANSAC_PX)
    if H is None or mask is None or not np.isfinite(H).all() or np.linalg.matrix_rank(H) != 3 or abs(H[2, 2]) < 1e-9:
        return None, np.zeros(len(p0), bool), None
    good = mask.ravel().astype(bool)
    if good.sum() < 4:
        return None, good, None
    H_norm = H / H[2, 2]
    predicted = cv2.perspectiveTransform(p1[good].reshape(-1, 1, 2), H_norm).reshape(-1, 2)
    rmse = float(np.sqrt(np.mean(np.sum((predicted - p0[good]) ** 2, axis=1))))
    return H_norm, good, rmse if np.isfinite(rmse) else None


def quad_points(shape):
    h, w = shape[:2]
    return np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)


def indicators(a, b, p0, p1, good, H):
    """Sáu chỉ báo rẻ: đếm/tỉ lệ điểm, phân tán không gian, vùng chồng, tương phản sau khi ghép, RMSE."""
    h, w = b.shape[:2]
    inliers = p0[good]
    out = {"matches": int(len(p0)), "inliers": int(good.sum()),
           "inlier_ratio": float(good.sum() / max(len(p0), 1))}
    hull = cv2.convexHull(np.float32(inliers)).reshape(-1, 2) if inliers.shape[0] >= 3 else None
    out["hull_area_ratio"] = float(cv2.contourArea(hull) / (h * w)) if hull is not None else 0.0
    inter, _ = cv2.intersectConvexConvex(quad_points(a.shape), cv2.perspectiveTransform(quad_points(b.shape), H))
    out["overlap_ratio"] = float(inter / float(w * h))
    gray_a = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gray_b = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY).astype(np.float32)
    inverse = np.linalg.inv(H)
    warped = cv2.warpPerspective(gray_a, inverse, (w, h))
    mask = cv2.warpPerspective(np.ones_like(gray_a), inverse, (w, h)) > 0.5
    x, y = warped[mask], gray_b[mask]
    if len(x) == 0:
        out["photo_ncc"] = 0.0
        out["photo_bad_frac"] = 1.0
        return out
    x, y = x - x.mean(), y - y.mean()
    out["photo_ncc"] = float((x * y).sum() / (np.sqrt((x * x).sum() * (y * y).sum()) + 1e-9))
    out["photo_bad_frac"] = float(np.mean(np.abs(x - y) > 25))
    return out


def marker_error(H_a, H_b, size):
    """Tính khoảng cách trung bình trên 5 điểm mốc giữa 2 homography; trả None nếu không hợp lệ."""
    if H_a is None or H_b is None:
        return None
    pts = np.float32([[0, 0], [size[0], 0], [size[0], size[1]], [0, size[1]],
                      [size[0] / 2, size[1] / 2]]).reshape(-1, 1, 2)
    a = cv2.perspectiveTransform(pts, H_a).reshape(-1, 2)
    b = cv2.perspectiveTransform(pts, H_b).reshape(-1, 2)
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        return None
    return float(np.mean(np.linalg.norm(a - b, axis=1)))


def estimate_cached(method, matcher, images, files, index_a, index_b, size_processed, size_original,
                    paths=None):
    """Đo một cặp bằng matcher; cache toàn cục để các chiến lược RQ3 dùng lại kết quả.

    Ghi nhận thời gian từng công đoạn rõ ràng và nhất quán:
    - io_seconds: thời gian đọc/giải mã ảnh từ đĩa (nếu chưa có trong cache)
    - match_seconds: thời gian chạy thuật toán tìm tương ứng (có sync CUDA)
    - geometry_seconds: thời gian RANSAC, homography, indicators và to_original
    - seconds: tổng thời gian thuật toán (match + geometry), không tính I/O.
    """
    path_a = str((paths or files)[index_a])
    path_b = str((paths or files)[index_b])
    key = (method, size_processed, path_a, path_b)
    if key in _MATCH_CACHE:
        return dict(_MATCH_CACHE[key], cache_hit=True)

    io_start = time.perf_counter()
    img_a = images[index_a]
    img_b = images[index_b]
    io_seconds = time.perf_counter() - io_start

    sync_device()
    match_start = time.perf_counter()
    p0, p1 = matcher(img_a, img_b)
    sync_device()
    match_seconds = time.perf_counter() - match_start

    geo_start = time.perf_counter()
    p0 = np.asarray(p0, np.float32).reshape(-1, 2)
    p1 = np.asarray(p1, np.float32).reshape(-1, 2)
    H, good, rmse = geometry(p0, p1)
    geo_seconds = time.perf_counter() - geo_start

    row = {
        "status": "success" if H is not None else "failed",
        "cache_hit": False,
        "io_seconds": io_seconds,
        "match_seconds": match_seconds,
        "geometry_seconds": geo_seconds,
        "seconds": match_seconds + geo_seconds,
        "rmse_px": rmse,
    }
    if H is not None:
        ind_start = time.perf_counter()
        row.update(indicators(img_a, img_b, p0, p1, good, H))
        row["H_processed"] = H.tolist()
        row["H_original"] = to_original(H, size_processed, size_original).tolist()
        ind_seconds = time.perf_counter() - ind_start
        row["geometry_seconds"] += ind_seconds
        row["seconds"] += ind_seconds

    _MATCH_CACHE[key] = row
    return dict(row)


def risk(row, tau_ncc=TAU_NCC, tau_ratio=TAU_INLIER_RATIO):
    """Rủi ro: matcher không ra H, tương phản sau khi ghép kém, hoặc quá ít inlier."""
    if row is None or row.get("status") != "success":
        return True
    ncc_risk = (row.get("photo_ncc") is None) or (row["photo_ncc"] < tau_ncc)
    ratio_risk = (row.get("inlier_ratio") is None) or (row["inlier_ratio"] < tau_ratio)
    return ncc_risk or ratio_risk


In [ ]:
# Đo cả cửa sổ: matcher tham chiếu cho từng bước + ba matcher, lưu mọi chỉ báo.
def scan_window(sequence, start, steps):
    files, lazy_images, size_original = frames_of(sequence, start, steps)
    steps = len(files) - 1
    size_processed = lazy_images.processed_size
    reference, chain = {}, [np.eye(3)]
    for step in range(1, steps + 1):
        H_ref, inliers, matches, seconds, valid = reference_pair(files[step - 1], files[step])
        reference[step] = {
            "H_original": H_ref.tolist() if H_ref is not None else None,
            "inliers": inliers,
            "matches": matches,
            "seconds": seconds,
            "valid": valid,
        }
        if chain[-1] is not None and valid and H_ref is not None:
            chain.append(chain[-1] @ H_ref)
        else:
            chain.append(None)

    matchers, rows = {}, {}
    for method in (CHEAP, STRONG, DIAGNOSTIC):
        matchers[method] = load_matcher(method, DEVICE)
        per_step = {}
        for step in range(1, steps + 1):
            row = dict(estimate_cached(method, matchers[method], lazy_images, files, step - 1, step,
                                       size_processed, size_original), step=step, tier=method)
            if row["status"] == "success" and reference[step]["valid"] and reference[step]["H_original"] is not None:
                H = to_original(np.array(row["H_processed"]), size_processed, size_original)
                row["step_error_px"] = marker_error(H, np.array(reference[step]["H_original"]), size_original)
            else:
                row["step_error_px"] = None
            per_step[step] = row
        rows[method] = per_step
        errors = [r["step_error_px"] for r in per_step.values() if r.get("step_error_px") is not None]
        med_err = np.median(errors) if errors else None
        max_err = np.max(errors) if errors else None
        print(f"  {method:22} step_err trung vị {fmt(med_err, 6)} | lớn nhất {fmt(max_err, 8)} px | "
              f"{sum(r['seconds'] for r in per_step.values()):6.1f}s | "
              f"lỗi hẳn {sum(1 for r in per_step.values() if r['status'] != 'success')} bước")

    weak = [step for step, value in reference.items() if not value["valid"]]
    if weak:
        print(f"  CẢNH BÁO: {len(weak)}/{steps} bước có tham chiếu dưới {MIN_INLIERS} inlier "
              f"(RDR-0007) -> đánh dấu không khả dụng, không đưa vào so sánh RQ1/RQ2/RQ3: {weak}")
    return {"sequence": sequence, "start": start, "steps": steps, "size_original": size_original,
            "size_processed": size_processed, "files": files, "lazy_images": lazy_images,
            "reference": reference,
            "reference_chain": [c.tolist() if c is not None else None for c in chain],
            "reference_invalid_steps": weak, "rows": rows}


In [ ]:
# RQ1: chỉ báo nào báo được bước ghép sai? (Spearman + AUC so với sai số thật của bước)
INDICATOR_NAMES = ("photo_ncc", "photo_bad_frac", "inlier_ratio", "inliers", "matches",
                   "hull_area_ratio", "overlap_ratio", "rmse_px")


def average_ranks(values):
    """Rank trung bình cho giá trị đồng hạng (đúng chuẩn Spearman; tránh rank giả theo thứ tự xuất hiện)."""
    values = np.asarray(values, float)
    ranks = np.empty(len(values), float)
    ranks[np.argsort(values, kind="mergesort")] = np.arange(1, len(values) + 1)
    unique, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)
    if len(unique) == len(values):
        return ranks
    inverse = inverse.reshape(-1)
    for index, count in enumerate(counts):
        if count > 1:
            ranks[inverse == index] = ranks[inverse == index].mean()
    return ranks


def spearman(x, y):
    """Spearman có xử lý đồng hạng; trả None nếu một biến không đổi (tương quan vô định)."""
    if len(x) < 2:
        return None
    rx = average_ranks(x) - (len(x) + 1) / 2
    ry = average_ranks(y) - (len(y) + 1) / 2
    denominator = np.sqrt((rx * rx).sum() * (ry * ry).sum())
    return float((rx * ry).sum() / denominator) if denominator > 0 else None


def auc(scores, labels):
    """AUC kiểu hạng, không cần sklearn; score lớn hơn = nghi ngờ hơn."""
    scores, labels = np.asarray(scores, float), np.asarray(labels, bool)
    if labels.all() or not labels.any():
        return None      # không có cả hai lớp -> AUC vô nghĩa, ghi None chứ không ghi NaN
    ranks = np.empty(len(scores), float)
    ranks[np.argsort(scores)] = np.arange(1, len(scores) + 1)
    for value, count in zip(*np.unique(scores, return_counts=True)):
        if count > 1:
            ranks[scores == value] = ranks[scores == value].mean()
    n_pos, n_neg = labels.sum(), (~labels).sum()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


# Chiều "nghi ngờ" của từng chỉ báo (điểm cao = dễ là bước ghép sai) khai báo TRƯỚC khi nhìn nhãn,
# theo ý nghĩa vật lý. Bản cũ lấy max(AUC, 1−AUC) sau khi nhìn nhãn = tự chọn chiều có lợi.
INDICATOR_SUSPICION = {"photo_ncc": -1.0, "photo_bad_frac": 1.0, "inlier_ratio": -1.0,
                       "inliers": -1.0, "matches": -1.0, "hull_area_ratio": -1.0,
                       "overlap_ratio": -1.0, "rmse_px": 1.0}


def rq1_table(scans, tolerance_px=TOLERANCE_PX):
    table = []
    for scan in scans:
        invalid = set(scan["reference_invalid_steps"])
        for tier, per_step in scan["rows"].items():
            # Chỉ chấm điểm bước có cả matcher chạy được LẪN tham chiếu hợp lệ (>= MIN_INLIERS)
            rows = [r for step, r in per_step.items()
                    if r.get("status") == "success" and step not in invalid and r.get("step_error_px") is not None]
            if not rows:
                continue
            labels = [r["step_error_px"] > tolerance_px for r in rows]
            for name in INDICATOR_NAMES:
                if name not in rows[0] or rows[0][name] is None:
                    continue
                values = [r[name] for r in rows]
                # AUC theo chiều đã khai báo ở trên: < 0.5 nghĩa là chỉ báo tách NGƯỢC chiều kỳ vọng.
                table.append({
                    "sequence": scan["sequence"], "tier": tier, "indicator": name,
                    "n_steps": len(rows), "n_bad_steps": int(sum(labels)),
                    "n_invalid_reference": len(invalid),
                    "spearman_vs_step_error": spearman(values, [r["step_error_px"] for r in rows]),
                    "auc_bad_step": auc([INDICATOR_SUSPICION[name] * v for v in values], labels)
                })
    return table


In [ ]:
# RQ2: mô phỏng chính sách phân luồng trên log đã đo (không chạy lại matcher).
def chain_from_steps(chosen, reference_chain):
    """Chuỗi H tích luỹ; dừng/lan truyền None ở bước thiếu H thay vì KeyError."""
    chain = [np.eye(3)]
    for position in range(1, len(reference_chain)):
        pick = chosen.get(position)
        if pick is None or pick.get("status") != "success" or "H_original" not in pick or pick["H_original"] is None or chain[-1] is None:
            chain.append(None)
        else:
            chain.append(chain[-1] @ np.array(pick["H_original"], float))
    return chain


def simulate(scan, policy, strong=STRONG, tau_ncc=TAU_NCC, tau_ratio=TAU_INLIER_RATIO, strong_rows=None):
    rows = scan["rows"]
    # An toàn khi strong không nằm trong rows
    strong_rows = strong_rows or rows.get(strong, {})
    size_original = scan["size_original"]
    reference_chain = [np.array(c, float) if c is not None else None for c in scan["reference_chain"]]
    chosen, seconds, escalations = {}, 0.0, 0
    for step in range(1, scan["steps"] + 1):
        cheap = rows[CHEAP][step]
        if policy == "classical":
            pick, run = cheap, [cheap]
        elif policy == "learned":
            pick, run = strong_rows[step], [strong_rows[step]]
        else:
            if risk(cheap, tau_ncc, tau_ratio):
                pick, run = strong_rows[step], [cheap, strong_rows[step]]
                escalations += 1
            else:
                pick, run = cheap, [cheap]
        seconds += sum(r["seconds"] for r in run)
        chosen[step] = pick

    chain = chain_from_steps(chosen, reference_chain)
    drift = []
    for i in range(len(chain)):
        if chain[i] is None or i >= len(reference_chain) or reference_chain[i] is None:
            drift.append(None)
        else:
            drift.append(marker_error(chain[i], reference_chain[i], size_original))

    # Loại trừ frame 0 (identity so với identity) khỏi thống kê drift thực tế
    non_root_drifts = [drift[i] for i in range(1, len(drift)) if drift[i] is not None]
    final_drift = non_root_drifts[-1] if non_root_drifts else None
    max_drift = float(np.max(non_root_drifts)) if non_root_drifts else None
    last_eval_frame = max([i for i in range(1, len(drift)) if drift[i] is not None], default=None)
    terminal_evaluable = (len(drift) > 1 and drift[-1] is not None)

    # Chỉ tính sai số bước tại các bước ghép thành công và có tham chiếu hợp lệ
    errors = [chosen[s]["step_error_px"] for s in sorted(chosen)
              if chosen[s].get("status") == "success" and chosen[s].get("step_error_px") is not None]
    return {
        "policy": policy,
        "tau_ncc": tau_ncc if policy == "routed" else None,
        "steps": len(chosen),
        "failed_steps": sum(1 for s in chosen if chosen[s].get("status") != "success"),
        "chained_steps": sum(1 for c in chain[1:] if c is not None),
        "reference_invalid_steps": len(scan["reference_invalid_steps"]),
        "escalations": escalations,
        "escalation_rate": escalations / max(len(chosen), 1),
        "seconds": seconds,
        "step_error_mean_px": float(np.mean(errors)) if errors else None,
        "step_error_p95_px": float(np.percentile(errors, 95)) if errors else None,
        "step_error_max_px": float(np.max(errors)) if errors else None,
        "final_drift_px": final_drift,
        "drift_max_px": max_drift,
        "last_evaluated_frame": last_eval_frame,
        "terminal_frame_evaluable": terminal_evaluable,
        "valid_drift_count": len(non_root_drifts),
        "unavailable_drift_count": max(0, (len(drift) - 1) - len(non_root_drifts)),
        "drift_series": drift,
    }


def choose_tau(sweep, budget=ESCALATION_BUDGET):
    """Ngưỡng LỚN NHẤT mà mọi sequence vẫn nằm trong ngân sách leo thang.

    Bảo toàn trạng thái infeasible (trả về None nếu không có ngưỡng nào đạt ngân sách),
    không tự ý thay bằng TAU_NCC.
    """
    routed = [row for row in sweep if row.get("policy") == "routed" and row.get("tau_ncc") is not None]
    if not routed:
        return None
    tau_vals = sorted({row["tau_ncc"] for row in routed})
    eligible = [tau for tau in tau_vals
                if all(row["escalation_rate"] <= budget for row in routed if row["tau_ncc"] == tau)]
    return max(eligible) if eligible else None

In [ ]:
# RQ3: pipeline online - so sánh 4 chiến lược phục hồi với matcher cố định.
class OnlinePipeline:
    def __init__(self, method, matcher, files, images, size_original, policy,
                 tau_ncc=TAU_NCC, tau_ratio=TAU_INLIER_RATIO, anchor_back=ANCHOR_BACK):
        self.method = method
        self.matcher = matcher
        self.files = files
        self.images = images
        self.size_original = size_original
        self.size_processed = (images[0].shape[1], images[0].shape[0]) if hasattr(images, '__getitem__') and len(images) > 0 else (640, 480)
        self.policy = policy
        self.tau_ncc = tau_ncc
        self.tau_ratio = tau_ratio
        self.anchor_back = anchor_back
        self.cost = 0.0
        self.triggers = self.skips = self.anchored = self.splits = 0
        self.reference_invalid_used = 0     # số lần đo gặp cặp tham chiếu không hợp lệ
        self.submosaic_of = {0: 0}
        self.submosaic_root = {0: 0}        # id submosaic -> CHỈ SỐ frame gốc của nó
        self.H_to_root = {0: np.eye(3)}
        self.steps = {}
        self.attempts = []
        self.action_log = {}

    def estimate(self, index_a, index_b):
        row = estimate_cached(self.method, self.matcher, self.images, self.files, index_a, index_b,
                              self.size_processed, self.size_original)
        self.cost += row["seconds"]          # tính chi phí kể cả khi lấy từ cache
        return row

    def step_error(self, previous, current, H_original):
        H_ref, inliers, _, _, valid = reference_pair(self.files[previous], self.files[current])
        if not valid or H_ref is None or H_original is None:
            self.reference_invalid_used += 1
            return None
        return marker_error(np.array(H_original, float), np.array(H_ref, float), self.size_original)

    def try_anchor(self, placed, index):
        """Neo vào các frame ĐÃ được đặt gần nhất, ghi nhận mọi attempt để kiểm chứng."""
        for anchor in reversed(placed[:-1][-self.anchor_back:]):
            candidate = self.estimate(anchor, index)
            is_risky = risk(candidate, self.tau_ncc, self.tau_ratio)
            self.attempts.append({
                "step": index, "pair": [anchor, index], "candidate_type": "anchor",
                "status": candidate.get("status"), "photo_ncc": candidate.get("photo_ncc"),
                "inlier_ratio": candidate.get("inlier_ratio"), "risk": is_risky,
                "accepted": not is_risky
            })
            if not is_risky:
                return dict(candidate, anchor=anchor)
        return None

    def run(self, steps):
        self.steps_total = steps
        placed, submosaic, index = [0], 0, 1
        self.action_log[0] = {"action": "root", "anchor": None, "submosaic": 0}
        while index <= steps:
            previous = placed[-1]
            row = self.estimate(previous, index)
            is_risky = risk(row, self.tau_ncc, self.tau_ratio)
            reason, used = "tier", row
            if is_risky:
                self.triggers += 1
                if self.policy == "none":
                    reason = "accept_risky"
                elif self.policy == "drop":
                    used, reason = None, "drop"
                elif self.policy == "split":
                    used, reason = None, "split"
                elif self.policy == "anchor":
                    used = self.try_anchor(placed, index)
                    reason = "anchor" if used else "drop"

            self.attempts.append({
                "step": index, "pair": [previous, index], "candidate_type": "consecutive",
                "status": row.get("status"), "photo_ncc": row.get("photo_ncc"),
                "inlier_ratio": row.get("inlier_ratio"), "risk": is_risky,
                "action": reason
            })

            if used is not None and used.get("status") == "success":
                root = used["anchor"] if reason == "anchor" else previous
                self.H_to_root[index] = self.H_to_root[root] @ np.array(used["H_original"], float)
                self.submosaic_of[index] = self.submosaic_of[root]
                self.steps[index] = {
                    "reason": reason, "anchor": root,
                    "step_error_px": self.step_error(root, index, used["H_original"])
                }
                placed.append(index)
                self.action_log[index] = {"action": reason, "anchor": root, "submosaic": self.submosaic_of[index]}
                if reason == "anchor":
                    self.anchored += 1
            elif reason == "split":
                submosaic += 1
                self.splits += 1
                self.H_to_root[index] = np.eye(3)
                self.submosaic_of[index] = submosaic
                self.submosaic_root[submosaic] = index
                placed.append(index)
                self.action_log[index] = {"action": "split", "anchor": index, "submosaic": submosaic}
            else:
                self.skips += 1
                self.action_log[index] = {"action": "drop", "anchor": None, "submosaic": None}
            index += 1
        self.placed_total = len(placed)
        return self

    def summary(self):
        placed = sorted(self.steps)
        errors = [self.steps[i]["step_error_px"] for i in placed if self.steps[i]["step_error_px"] is not None]

        # Tích luỹ chuỗi tham chiếu từng mảng O(n) thay vì O(n^2), lan truyền unknown khi tham chiếu hỏng
        drift = {}
        by_submosaic = {}
        for idx in range(self.steps_total + 1):
            if idx in self.submosaic_of:
                by_submosaic.setdefault(self.submosaic_of[idx], []).append(idx)

        for sub, f_indices in by_submosaic.items():
            root = self.submosaic_root[sub]
            step_ref_chain = {root: np.eye(3)}
            ref_accum = np.eye(3)
            ref_valid = True
            max_f = max(f_indices)
            for s in range(root + 1, max_f + 1):
                H_ref, inliers, _, _, valid = reference_pair(self.files[s - 1], self.files[s])
                if not valid or H_ref is None or not ref_valid:
                    ref_valid = False
                    step_ref_chain[s] = None
                else:
                    ref_accum = ref_accum @ np.array(H_ref, float)
                    step_ref_chain[s] = ref_accum.copy()

            for idx in f_indices:
                if idx == root:
                    drift[idx] = 0.0
                else:
                    ref_chain = step_ref_chain.get(idx)
                    if ref_chain is None or self.H_to_root.get(idx) is None:
                        drift[idx] = None
                    else:
                        drift[idx] = marker_error(self.H_to_root[idx], ref_chain, self.size_original)

        # Loại trừ identity root khỏi thống kê drift thực tế
        placed_non_roots = [p for p in placed if p > 0 and drift.get(p) is not None]
        valid_drifts = [drift[p] for p in placed_non_roots]
        last_eval_frame = placed_non_roots[-1] if placed_non_roots else None
        terminal_evaluable = (len(placed) > 0 and drift.get(placed[-1]) is not None and placed[-1] > 0)

        # Bản ghi chi tiết từng frame để replay và kiểm chứng độc lập
        frame_records = []
        for f in range(self.steps_total + 1):
            is_placed = f in self.submosaic_of
            frame_records.append({
                "frame_id": f,
                "placed": is_placed,
                "action": self.action_log.get(f, {}).get("action", "skipped"),
                "anchor": self.action_log.get(f, {}).get("anchor"),
                "submosaic_id": self.submosaic_of.get(f),
                "submosaic_root": self.submosaic_root.get(self.submosaic_of.get(f, -1)),
                "H_to_root": self.H_to_root[f].tolist() if f in self.H_to_root else None,
                "step_error_px": self.steps[f]["step_error_px"] if f in self.steps else None,
                "drift_px": drift.get(f),
            })

        return {
            "policy": self.policy,
            "matcher": self.method,
            "reference_invalid_used": self.reference_invalid_used,
            "frames_placed": self.placed_total,
            "frames_requested": self.steps_total + 1,
            "frames_with_step": len(placed),
            "submosaics": self.splits + 1,
            "triggers": self.triggers,
            "skipped": self.skips,
            "anchored": self.anchored,
            "seconds": self.cost,
            "step_error_mean_px": float(np.mean(errors)) if errors else None,
            "step_error_p95_px": float(np.percentile(errors, 95)) if errors else None,
            "step_error_max_px": float(np.max(errors)) if errors else None,
            "drift_series": [drift.get(i) for i in sorted(self.submosaic_of)],
            "drift_max_px": float(np.max(valid_drifts)) if valid_drifts else None,
            "drift_final_px": valid_drifts[-1] if valid_drifts else None,
            "last_evaluated_frame": last_eval_frame,
            "terminal_frame_evaluable": terminal_evaluable,
            "valid_drift_count": len(placed_non_roots),
            "unavailable_drift_count": max(0, (len(placed) - 1) - len(placed_non_roots)),
            "records": frame_records,
            "attempts": self.attempts,
        }


def run_policies(scan, method, tau_ncc, policies=POLICIES):
    matcher = load_matcher(method, DEVICE)
    out = []
    for policy in policies:
        pipeline = OnlinePipeline(method, matcher, scan["files"], scan["lazy_images"],
                                  scan["size_original"], policy, tau_ncc=tau_ncc).run(scan["steps"])
        summary = pipeline.summary()
        out.append({"sequence": scan["sequence"], **summary})
        print(f"  {method:20} {policy:7} đặt {summary['frames_placed']:3}/{scan['steps'] + 1} frame | "
              f"trigger {summary['triggers']:2} | submosaic {summary['submosaics']} | "
              f"step_err max {fmt(summary['step_error_max_px'], 8)} | drift max {fmt(summary['drift_max_px'], 9)} px")
    return out


In [ ]:
# Chạy toàn bộ: RQ1 trên các sequence được cấu hình.
torch.set_num_threads(4)
cv2.setNumThreads(4)
torch.manual_seed(0)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

# Tạo thư mục chạy duy nhất theo STAMP; không bao giờ ghi đè kết quả cũ
TARGET = OUTPUT_ROOT / "failure_aware_pipeline" / STAMP
if TARGET.is_dir() and (TARGET / "completed.json").is_file():
    raise RuntimeError(f"Target directory already completed: {TARGET}. Run must use a fresh unique timestamp.")
TARGET.mkdir(parents=True, exist_ok=False)

# Xoá sạch cache toàn cục khi bắt đầu run mới để đảm bảo tính độc lập
_MATCH_CACHE.clear()
_REFERENCE_CACHE.clear()

REPORT = {
    "stamp": STAMP,
    "scope": "RQ1/RQ2/RQ3 - Failure-Aware Mosaicking Technical Pipeline (LOSO Evaluation)",
    "execution_mode": EXECUTION_MODE,
    "manifest": {
        "platform": platform.platform(), "torch": torch.__version__,
        "opencv": cv2.__version__, "numpy": np.__version__, "device": DEVICE,
        "max_side": MAX_SIDE, "max_keypoints": MAX_KEYPOINTS, "ransac_px": RANSAC_PX,
        "ratio": RATIO, "min_inliers": MIN_INLIERS, "tau_grid": list(TAU_GRID),
        "escalation_budget": ESCALATION_BUDGET, "anchor_back": ANCHOR_BACK,
        "seed": 0, "windows": {s: list(WINDOWS[s]) for s in SEQUENCES},
        "cheap": CHEAP, "strong": STRONG, "diagnostic": DIAGNOSTIC, "fullres": FULLRES,
        "tau_ncc_default": TAU_NCC, "tau_inlier_ratio_default": TAU_INLIER_RATIO,
        "tolerance_px": TOLERANCE_PX,
        "protocol": "Leave-One-Sequence-Out (LOSO) per RDR-0011; village/grass have historical indicator discovery limitation.",
        "note": "Ngưỡng số trị TAU_* mang tính chất khám phá (exploratory) theo RDR-0008 (Proposed)."
    },
    "scans": [], "rq1": [], "rq2": [], "rq3": [], "loso_folds": {}, "within_sequence_diagnostics": {},
    "inputs": []
}

SCANS = []
for sequence in SEQUENCES:
    start, steps = WINDOWS[sequence]
    step_label = f"{steps} steps" if steps is not None else "full sequence"
    print(f"\n=== {sequence}: frame {start} ({step_label})")
    scan = scan_window(sequence, start, steps)
    SCANS.append(scan)
    REPORT["inputs"].extend(
        {"path": str(p), "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
         "processed_hw": [scan["size_processed"][1], scan["size_processed"][0]]}
        for p in scan["files"]
    )

REPORT["rq1"] = rq1_table(SCANS)
print(f"\nRQ1: {len(REPORT['rq1'])} dòng chỉ báo đã tính (đã loại bỏ toàn bộ bước tham chiếu không hợp lệ)")

In [ ]:
# RQ2 & RQ3: Đánh giá theo giao thức Leave-One-Sequence-Out (LOSO) chuẩn RDR-0011.
# Ngưỡng tau* được hiệu chỉnh CHỈ trên tập train (các sequence khác), sau đó áp dụng vào
# phân luồng, leo thang ảnh gốc, và 4 chiến lược phục hồi của sequence kiểm thử giữ lại.

# 1. Quét lưới khám phá nội bộ sequence (chẩn đoán tham khảo)
scan_sweeps = {}
for scan in SCANS:
    seq = scan["sequence"]
    sweep = [simulate(scan, "classical"), simulate(scan, "learned")]
    sweep += [simulate(scan, "routed", tau_ncc=t) for t in TAU_GRID]
    scan_sweeps[seq] = sweep
    REPORT["within_sequence_diagnostics"][seq] = sweep

# 2. Vòng lặp LOSO Fold Execution
REPORT["loso_folds"] = {}
REPORT["rq2"] = []
REPORT["rq3"] = []

for test_idx, test_scan in enumerate(SCANS):
    test_seq = test_scan["sequence"]
    train_scans = [s for i, s in enumerate(SCANS) if i != test_idx]
    print(f"\n>>> LOSO Fold: Test = {test_seq} | Train = {[s['sequence'] for s in train_scans]}")

    if not train_scans:
        fold_status = "not_evaluable"
        tau_calibrated = None
        eval_tau = TAU_NCC
        tau_note = "Không có sequence train; dùng ngưỡng khám phá mặc định"
    else:
        train_sweeps = []
        for s in train_scans:
            train_sweeps.extend([r for r in scan_sweeps[s["sequence"]] if r.get("policy") == "routed"])
        tau_calibrated = choose_tau(train_sweeps, budget=ESCALATION_BUDGET)
        if tau_calibrated is not None:
            fold_status = "budget_compliant"
            eval_tau = tau_calibrated
            tau_note = f"Hiệu chỉnh thành công từ train fold: τ_ncc = {tau_calibrated}"
        else:
            fold_status = "infeasible"
            eval_tau = TAU_NCC
            tau_note = f"Train fold không đạt ngân sách {ESCALATION_BUDGET*100:.0f}%; dùng ngưỡng dự phòng"

    print(f"  Trạng thái fold: {fold_status} | Ngưỡng áp dụng: τ_ncc = {eval_tau} ({tau_note})")

    # Đánh giá RQ2 Held-out trên test_scan
    test_classical = simulate(test_scan, "classical")
    test_learned = simulate(test_scan, "learned")
    test_routed = simulate(test_scan, "routed", tau_ncc=eval_tau)
    REPORT["rq2"].extend([
        {"sequence": test_seq, "fold": test_seq, **test_classical},
        {"sequence": test_seq, "fold": test_seq, **test_learned},
        {"sequence": test_seq, "fold": test_seq, "tau_calibrated": tau_calibrated, "status": fold_status, **test_routed}
    ])

    # Leo thang ảnh gốc (FULLRES) tại các bước bị gắn cờ bởi eval_tau
    flagged = [s for s in range(1, test_scan["steps"] + 1)
               if risk(test_scan["rows"][CHEAP][s], eval_tau, TAU_INLIER_RATIO)]
    matcher = load_matcher(STRONG, DEVICE)
    full_images = LazyImages(test_scan["files"], max_cached=16)
    full = {}
    for step in flagged:
        row = estimate_cached(FULLRES, matcher, full_images, test_scan["files"], step - 1, step,
                              test_scan["size_original"], test_scan["size_original"], paths=test_scan["files"])
        row = dict(row, step=step, tier=FULLRES)
        if row["status"] == "success" and test_scan["reference"][step]["valid"] and test_scan["reference"][step]["H_original"] is not None:
            row["step_error_px"] = marker_error(np.array(row["H_original"], float),
                                                np.array(test_scan["reference"][step]["H_original"], float),
                                                test_scan["size_original"])
        else:
            row["step_error_px"] = None
        full[step] = row

    fullres_rows = {step: (full[step] if step in full else dict(test_scan["rows"][STRONG][step], tier=FULLRES))
                    for step in range(1, test_scan["steps"] + 1)}
    test_scan["rows"][FULLRES] = fullres_rows
    del matcher

    test_routed_full = simulate(test_scan, "routed", tau_ncc=eval_tau, strong_rows=fullres_rows)
    REPORT["rq2"].append({"sequence": test_seq, "fold": test_seq, "tau_calibrated": tau_calibrated,
                          "status": fold_status, **test_routed_full, "policy": "routed_fullres"})

    # Đánh giá RQ3 Recovery Policies dưới cùng ngưỡng eval_tau của fold
    rq3_method = RQ3_MATCHER_MAP.get(test_seq, CHEAP)
    rq3_summaries = run_policies(test_scan, rq3_method, eval_tau)
    for r in rq3_summaries:
        r["fold"] = test_seq
        r["tau_calibrated"] = tau_calibrated
        r["status"] = fold_status
    REPORT["rq3"].extend(rq3_summaries)

    REPORT["loso_folds"][test_seq] = {
        "test_sequence": test_seq,
        "train_sequences": [s["sequence"] for s in train_scans],
        "tau_calibrated": tau_calibrated,
        "fold_status": fold_status,
        "eval_tau_used": eval_tau,
        "classical": test_classical,
        "learned": test_learned,
        "routed": test_routed,
        "routed_fullres": test_routed_full,
        "rq3": rq3_summaries,
    }

# Cập nhật REPORT['scans'] để đảm bảo dữ liệu scan xuất ra đồng bộ hoàn toàn với FULLRES
REPORT["scans"] = []
for scan in SCANS:
    REPORT["scans"].append({
        "sequence": scan["sequence"], "start": scan["start"], "steps": scan["steps"],
        "size_original": scan["size_original"], "size_processed": scan["size_processed"],
        "reference_inliers_min": min((v["inliers"] for v in scan["reference"].values()), default=0),
        "reference_invalid_steps": scan["reference_invalid_steps"],
        "reference": {str(k): v for k, v in scan["reference"].items()},
        "rows": {t: {str(k): v for k, v in scan["rows"][t].items()} for t in scan["rows"]}
    })
print(f"\nĐã hoàn thành đánh giá LOSO cho {len(REPORT['loso_folds'])} folds.")


In [ ]:
# RQ3 Summary & Khẳng định cấu hình: Ghi nhận kết quả tổng hợp của giao thức LOSO.
print(f"Tổng kết RQ3 trên toàn bộ {len(REPORT['loso_folds'])} fold LOSO:")
for test_seq, fold_res in REPORT["loso_folds"].items():
    print(f"Fold [{test_seq}]: τ_calibrated = {fold_res['tau_calibrated']} | trạng thái = {fold_res['fold_status']}")
    for pol in fold_res["rq3"]:
        print(f"  {pol['policy']:7} đặt {pol['frames_placed']:2}/{pol['frames_requested']:2} frame | "
              f"triggers {pol['triggers']:2} | submosaics {pol['submosaics']:2} | "
              f"drift max {fmt(pol['drift_max_px'], 8)} px | terminal_evaluable = {pol['terminal_frame_evaluable']}")


In [ ]:
# Tự kiểm tra tính đúng đắn (Deterministic Cloud Self-Checks & Invariants)

# (A) Kiểm tra thực thi chính sách Split và tính toán drift gốc khác 0 (nonzero root)
test_files = [Path(f"synth_f{i}.jpg") for i in range(5)]
def make_test_T(tx, ty):
    return np.array([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]], dtype=float)

# Mock dữ liệu cặp ảnh vào cache để chạy thật OnlinePipeline.run() và summary()
_MATCH_CACHE.clear()
_REFERENCE_CACHE.clear()

# Bước 1 (0 -> 1): an toàn (photo_ncc = 0.99)
_MATCH_CACHE[("synth_test", (640, 480), str(test_files[0]), str(test_files[1]))] = {
    "status": "success", "seconds": 0.01, "rmse_px": 0.5, "photo_ncc": 0.99, "inlier_ratio": 0.6,
    "H_processed": make_test_T(10, 0).tolist(), "H_original": make_test_T(10, 0).tolist()
}
_REFERENCE_CACHE[(str(test_files[0]), str(test_files[1]))] = (make_test_T(10, 0), 200, 250, 0.01, True)

# Bước 2 (1 -> 2): rủi ro cao (photo_ncc = 0.50) -> kích hoạt split tại frame 2!
_MATCH_CACHE[("synth_test", (640, 480), str(test_files[1]), str(test_files[2]))] = {
    "status": "success", "seconds": 0.01, "rmse_px": 2.5, "photo_ncc": 0.50, "inlier_ratio": 0.1,
    "H_processed": make_test_T(10, 0).tolist(), "H_original": make_test_T(10, 0).tolist()
}
_REFERENCE_CACHE[(str(test_files[1]), str(test_files[2]))] = (make_test_T(10, 0), 200, 250, 0.01, True)

# Bước 3 (2 -> 3): an toàn sau split (gốc mới là frame 2, bước 2->3 dịch chuyển 5px)
_MATCH_CACHE[("synth_test", (640, 480), str(test_files[2]), str(test_files[3]))] = {
    "status": "success", "seconds": 0.01, "rmse_px": 0.5, "photo_ncc": 0.99, "inlier_ratio": 0.6,
    "H_processed": make_test_T(5, 0).tolist(), "H_original": make_test_T(5, 0).tolist()
}
_REFERENCE_CACHE[(str(test_files[2]), str(test_files[3]))] = (make_test_T(5, 0), 200, 250, 0.01, True)

# Bước 4 (3 -> 4): an toàn sau split
_MATCH_CACHE[("synth_test", (640, 480), str(test_files[3]), str(test_files[4]))] = {
    "status": "success", "seconds": 0.01, "rmse_px": 0.5, "photo_ncc": 0.99, "inlier_ratio": 0.6,
    "H_processed": make_test_T(5, 0).tolist(), "H_original": make_test_T(5, 0).tolist()
}
_REFERENCE_CACHE[(str(test_files[3]), str(test_files[4]))] = (make_test_T(5, 0), 200, 250, 0.01, True)

mock_imgs = [np.zeros((480, 640, 3), np.uint8)] * 5
synth_pipe = OnlinePipeline("synth_test", None, test_files, mock_imgs, (640, 480), "split", tau_ncc=0.95).run(4)
synth_summary = synth_pipe.summary()

assert synth_pipe.splits == 1, "Chính sách split phải thực sự tách mảng"
assert synth_summary["submosaics"] == 2, "Phải tạo ra đúng 2 submosaics"
assert synth_pipe.submosaic_root[1] == 2, "Gốc submosaic mới phải là frame 2"
assert synth_summary["drift_series"][3] < 1e-5, "Drift frame 3 phải đo từ gốc mới (frame 2), không được đo từ frame 0"
assert synth_summary["drift_series"][4] < 1e-5, "Drift frame 4 phải đo từ gốc mới"
assert len(synth_summary["attempts"]) >= 4, "Phải lưu nhật ký attempt"

# (B) Kiểm tra chuỗi tham chiếu hoàn toàn không hợp lệ (All-Invalid Reference Check)
mock_invalid_scan = {
    "sequence": "mock_all_invalid", "steps": 3, "size_original": (640, 480),
    "reference_chain": [np.eye(3).tolist(), None, None, None],
    "reference_invalid_steps": [1, 2, 3],
    "rows": {
        CHEAP: {
            1: {"status": "success", "seconds": 0.01, "H_original": make_test_T(10, 0).tolist(), "step_error_px": None},
            2: {"status": "success", "seconds": 0.01, "H_original": make_test_T(10, 0).tolist(), "step_error_px": None},
            3: {"status": "success", "seconds": 0.01, "H_original": make_test_T(10, 0).tolist(), "step_error_px": None},
        }
    }
}
invalid_sim = simulate(mock_invalid_scan, "classical")
assert invalid_sim["final_drift_px"] is None, "Chuỗi tham chiếu hỏng hoàn toàn không được báo drift bằng 0.0"
assert invalid_sim["valid_drift_count"] == 0, "Không có phép đo drift nào hợp lệ"
assert invalid_sim["unavailable_drift_count"] == 3
assert invalid_sim["terminal_frame_evaluable"] is False

# (C) Kiểm tra tính độc lập của tập kiểm thử trong LOSO (Held-out Independence Check)
train_sample = [
    {"policy": "routed", "tau_ncc": 0.93, "escalation_rate": 0.15},
    {"policy": "routed", "tau_ncc": 0.94, "escalation_rate": 0.20},
    {"policy": "routed", "tau_ncc": 0.95, "escalation_rate": 0.35},
]
assert choose_tau(train_sample, budget=0.25) == 0.94
# Thay đổi dữ liệu test không được ảnh hưởng đến kết quả train
test_sample_bad = [{"policy": "routed", "tau_ncc": 0.90, "escalation_rate": 0.99}]
assert choose_tau(train_sample, budget=0.25) == 0.94
# Kiểm tra trường hợp infeasible: khi toàn bộ train vượt budget
infeasible_sample = [{"policy": "routed", "tau_ncc": t, "escalation_rate": 0.50} for t in TAU_GRID]
assert choose_tau(infeasible_sample, budget=0.25) is None

# (D) Kiểm tra tính toàn vẹn cấu trúc kết quả LOSO trên dữ liệu thực
assert len(REPORT["loso_folds"]) == len(SCANS)
for seq, fold in REPORT["loso_folds"].items():
    assert "classical" in fold and "learned" in fold and "routed" in fold and "rq3" in fold
    assert len(fold["rq3"]) == len(POLICIES)
    for r in fold["rq3"]:
        assert "records" in r and "attempts" in r

print("PASS: Toàn bộ kiểm tra thực thi Split, tham chiếu không hợp lệ và độc lập LOSO đều hợp lệ.")


In [ ]:
# Biểu đồ: (trái) chỉ báo vs sai số thật; (phải) drift của 4 chiến lược phục hồi theo giao thức LOSO.
figure, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 0: Phân bố chỉ báo quang học vs sai số bước
for tier, marker in ((CHEAP, "o"), (STRONG, "s"), (DIAGNOSTIC, "^")):
    for scan in SCANS:
        rows = [r for r in scan["rows"][tier].values()
                if r.get("status") == "success" and r.get("step_error_px") is not None]
        if rows:
            axes[0].scatter([r["photo_ncc"] for r in rows], [max(r["step_error_px"], 0.05) for r in rows],
                            s=18, marker=marker, alpha=0.75,
                            label=f"{tier} ({scan['sequence'].split('-')[1]})")

axes[0].axhline(TOLERANCE_PX, ls="--", color="k", lw=1, label=f"ngưỡng sai {TOLERANCE_PX:.0f} px")
axes[0].set_yscale("log")
axes[0].set_xlabel("tương phản sau khi ghép (photo_ncc)")
axes[0].set_ylabel("sai số bước so với SIFT tham chiếu (px, log)")
axes[0].set_title("RQ1: Phân bố chỉ báo quang học vs sai số bước")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=.3)

# Panel 1: RQ3 drift series cho toàn bộ sequence (không giới hạn số lượng)
color_cycle = plt.cm.tab10(np.linspace(0, 1, max(10, len(SCANS))))
styles = ["-", "--", "-.", ":"]

for s_idx, scan in enumerate(SCANS):
    sequence = scan["sequence"]
    scan_start = scan["start"]
    seq_color = color_cycle[s_idx % len(color_cycle)]
    fold_rq3 = REPORT["loso_folds"].get(sequence, {}).get("rq3", [])
    for p_idx, row in enumerate(fold_rq3):
        policy_style = styles[p_idx % len(styles)]
        records = row.get("records", [])
        by_sub = {}
        for rec in records:
            if rec.get("placed") and rec.get("submosaic_id") is not None:
                by_sub.setdefault(rec["submosaic_id"], []).append(rec)
        lbl_shown = False
        for sub_id, sub_recs in by_sub.items():
            plot_recs = [r for r in sub_recs if r["frame_id"] > sub_recs[0]["frame_id"] and r.get("drift_px") is not None]
            if not plot_recs:
                continue
            x_vals = [scan_start + r["frame_id"] for r in plot_recs]
            y_vals = [r["drift_px"] for r in plot_recs]
            label = f"{sequence.split('-')[1]} · {row['policy']}" if not lbl_shown else None
            axes[1].plot(x_vals, y_vals, policy_style, color=seq_color, lw=1.6, label=label)
            lbl_shown = True

axes[1].set_xlabel("Chỉ số frame video thực tế (real frame index)")
axes[1].set_ylabel("Độ trôi tích lũy nội bộ từng mảng (px)")
axes[1].set_title("RQ3: Bốn chiến lược phục hồi (ngắt đoạn tại split, loại trừ root 0 px)")
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(alpha=.3)
plt.tight_layout()
figure.savefig(TARGET / "summary.png", dpi=110)
plt.show()

## Hướng dẫn đọc và phân tích 3 bảng kết quả thực nghiệm theo giao thức LOSO

### 1. Bảng RQ1: Khả năng dự báo lỗi của 8 chỉ số tính nhanh
- **Hệ số tương quan Spearman (rho):** Đo mức độ ăn khớp thứ hạng giữa chỉ số cảnh báo và sai số bước ghép so với SIFT tham chiếu chất lượng cao ở ảnh gốc. Giá trị càng gần 1 hoặc -1 thể hiện độ nhạy càng cao.
- **Điểm phân loại AUC (Area Under Curve):** Thể hiện khả năng phân biệt giữa bước ghép đạt chuẩn và bước ghép sai lệch. Chiều nghi ngờ được khai báo trước theo ý nghĩa vật lý.
  * *Quan sát thực nghiệm:* Trên các cửa sổ thử nghiệm, chỉ số đếm điểm khớp (Inliers) có thể bị suy giảm độ tin cậy ở vùng địa hình lặp lại do tạo ra tương ứng giả. Chỉ báo quang học `photo_ncc` (tương quan độ sáng sau khi nắn hình học) cho thấy độ nhạy cao trong việc tách biệt các ca sai lệch hình học lớn.
- Cột `bước sai`: Số bước có sai số vượt quá 2 pixel (mức nền nhiễu cơ sở của SIFT tham chiếu theo đề xuất RDR-0008).

### 2. Bảng RQ2: Phân luồng thích ứng theo giao thức Leave-One-Sequence-Out (LOSO)
- **Hiệu chỉnh ngoài tập kiểm thử (Training-only Calibration):** Ngưỡng $\tau$ được chọn dựa trên dữ liệu các sequence huấn luyện trong ngân sách leo thang (25%). Sequence kiểm thử giữ lại không được dùng để tối ưu hóa ngưỡng.
- **Tỷ lệ leo thang và Độ trôi:** Đánh giá tiềm năng tiết kiệm thời gian tính toán của cơ chế định tuyến thích ứng so với việc luôn kích hoạt mô hình học sâu mạnh.

### 3. Bảng RQ3: So sánh 4 chiến lược phục hồi sự cố trên cùng ngưỡng của fold
- **None (Mặc kệ):** Tiếp tục ghép nối tiếp bất chấp cảnh báo rủi ro -> Sai lệch tích lũy qua chuỗi nhân ma trận làm méo mó bản đồ.
- **Drop (Bỏ qua khung hình):** Vứt bỏ ảnh có bước ghép lỗi -> Nguy cơ gây phản ứng dây chuyền làm mất liên tục nhiều khung hình tiếp theo nếu cảnh khó kéo dài.
- **Anchor (Ghép lùi về ảnh mốc cũ):** Cố tìm ảnh cũ trong lịch sử đã đặt để ghép vượt qua điểm nghẽn -> Hiệu quả phụ thuộc vào vùng chồng với các frame trước đó.
- **Split (Tách mảng bản đồ con):** Đóng mảng bản đồ hiện tại, khởi tạo mảng bản đồ mới tại khung hình đó -> Bảo toàn khung hình trong từng submosaic độc lập và ngắt tích lũy trôi trong nội bộ mảng; tuy nhiên vị trí tương đối giữa các mảng con không được xác định nếu thiếu thông tin định vị toàn cầu.


In [ ]:
rq1 = sorted(REPORT["rq1"], key=lambda r: (r["sequence"], r["tier"],
 -abs(r["spearman_vs_step_error"] or 0.0)))
print(f"{'sequence':22} {'matcher':22} {'chỉ báo':16} {'ρ':>7} {'AUC':>6} {'bước':>5} {'sai':>5} {'loại':>5}")
for row in rq1:
    score = "   n/a" if row["auc_bad_step"] is None else f"{row['auc_bad_step']:6.3f}"
    print(f"{row['sequence']:22} {row['tier']:22} {row['indicator']:16} "
          f"{fmt(row['spearman_vs_step_error'], 7)} {score} {row['n_steps']:5} {row['n_bad_steps']:5} "
          f"{row['n_invalid_reference']:5}")
print("(cột 'loại' = số bước bị loại vì cặp tham chiếu không hợp lệ hoặc dưới 100 inlier)")
print("(AUC 'n/a' = không có đủ cả bước tốt lẫn bước sai để tính AUC)")

print("\n=== KẾT QUẢ RQ2: ĐỊNH TUYẾN THEO GIAO THỨC LOSO (HELD-OUT TEST SEQUENCES) ===")
header = f"{'fold/test_seq':20} {'chính sách':16} {'τ_calib':>8} {'leo thang':>9} {'giây':>8} {'err TB':>8} {'drift max':>10} {'eval_last':>9}"
print(header)
for fold_seq, fold_res in REPORT["loso_folds"].items():
    for pol_name in ("classical", "learned", "routed", "routed_fullres"):
        r = fold_res.get(pol_name)
        if not r:
            continue
        label = r["policy"]
        tau_str = fmt(r.get("tau_ncc"), 8) if r.get("tau_ncc") is not None else "   n/a"
        eval_last = str(r.get("last_evaluated_frame")) if r.get("last_evaluated_frame") is not None else "None"
        print(f"{fold_seq:20} {label:16} {tau_str:>8} "
              f"{r['escalation_rate']:9.2f} {r['seconds']:8.1f} {fmt(r['step_error_mean_px'], 8)} "
              f"{fmt(r['drift_max_px'], 10)} {eval_last:>9}")

print("\n=== KẾT QUẢ RQ3: PHỤC HỒI THEO GIAO THỨC LOSO (HELD-OUT TEST SEQUENCES) ===")
header = f"{'fold/test_seq':20} {'matcher':20} {'chiến lược':10} {'frame':>9} {'sub':>4} {'trig':>5} {'err max':>9} {'drift max':>10} {'term_eval':>9}"
print(header)
for fold_seq, fold_res in REPORT["loso_folds"].items():
    for pol in fold_res["rq3"]:
        term_str = "True" if pol["terminal_frame_evaluable"] else "False"
        print(f"{fold_seq:20} {pol['matcher']:20} {pol['policy']:10} "
              f"{pol['frames_placed']:4}/{pol['frames_requested']:<4} "
              f"{pol['submosaics']:4} {pol['triggers']:5} {fmt(pol['step_error_max_px'], 9)} "
              f"{fmt(pol['drift_max_px'], 10)} {term_str:>9}")

# Lưu artifact cuối cùng: đảm bảo mọi dữ liệu đã tính xong hoàn chỉnh
for scan in SCANS:
    dump(TARGET / f"scan_{scan['sequence']}.json", {
        "sequence": scan["sequence"], "start": scan["start"], "steps": scan["steps"],
        "size_original": scan["size_original"], "size_processed": scan["size_processed"],
        "reference_inliers_min": min((v["inliers"] for v in scan["reference"].values()), default=0),
        "reference_invalid_steps": scan["reference_invalid_steps"],
        "reference": {str(k): v for k, v in scan["reference"].items()},
        "rows": {t: {str(k): v for k, v in scan["rows"][t].items()} for t in scan["rows"]}
    })

dump(TARGET / "report.json", REPORT)

# Ghi completed.json CUỐI CÙNG sau khi toàn bộ artifact đã hoàn tất
completion_marker = {
    "stamp": STAMP,
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "status": "success",
    "target": str(TARGET),
    "artifacts": [
        "report.json",
        "summary.png",
        *[f"scan_{s['sequence']}.json" for s in SCANS]
    ]
}
dump(TARGET / "completed.json", completion_marker)

try:
    target_display = TARGET.relative_to(ROOT)
except ValueError:
    target_display = TARGET
display(Markdown(f"**Đã lưu toàn bộ artifact thành công:** `{target_display}` (kèm `completed.json`)"))


## Các giới hạn nghiên cứu cần lưu ý khi báo cáo

1. **Thước đo sai số chuẩn là sự đồng thuận với matcher tham chiếu:** Do không có thiết bị đo đạc mặt đất chuyên dụng (RTK/GCP) làm ground truth thực địa, sai số thật của từng bước được đối chiếu qua thuật toán SIFT tham chiếu chất lượng cao ở độ phân giải gốc theo quy định tại `RDR-0007`.
2. **Giao thức Leave-One-Sequence-Out (LOSO) theo RDR-0011:** Toàn bộ ngưỡng kích hoạt cảnh báo rủi ro ($\tau_{ncc} = 0,93$, tolerance 2 px) mang tính chất khám phá (exploratory) theo `RDR-0008` (Proposed). Giao thức đánh giá chuẩn yêu cầu chia fold theo sequence để kiểm thử độc lập mà không nhìn trước nhãn của sequence đích.
3. **Giới hạn lịch sử phát triển phương pháp:** Hai chuỗi `phantom3-village-kfs` và `phantom3-grass-kfs` đã được sử dụng trong giai đoạn ban đầu để khám phá các chỉ báo và chọn cửa sổ thực nghiệm, do đó không thể coi các fold này hoàn toàn độc lập với quá trình hình thành đề xuất phương pháp.
4. **Đặc tính phục hồi Split ở RQ3:** Chiến lược tách submosaic (`split`) giúp bảo toàn khung hình và ngăn ngừa tích lũy sai số trôi trong từng mảng con độc lập, nhưng không ước lượng vị trí tương đối giữa các mảng con nếu không có dữ liệu định vị toàn cầu bổ sung.
5. **Cổng kiểm soát chất lượng:** Cặp tham chiếu dưới 100 inliers hoặc RANSAC thất bại luôn được ghi nhận rõ ràng là không khả dụng (`valid = False`), loại bỏ hoàn toàn khỏi các phép so sánh RQ1, RQ2 và RQ3 để không làm sai lệch số liệu khoa học.
